# Core Concepts: Grids and Fields

Optical simulations require representing continuous physical quantities on discrete computational grids. HCIPy provides two fundamental abstractions for this purpose:

* **Grids** define the spatial sampling points where calculations are performed
* **Fields** store physical quantities such as electric field amplitude, intensity, or phase at those points

Understanding these concepts is essential for all HCIPy simulations, from simple aperture calculations to complex adaptive optics systems. This tutorial provides a comprehensive introduction to creating and manipulating grids and fields.

In [ ]:
from hcipy import *
import numpy as np
import matplotlib.pyplot as plt

## Grids

A `Grid` defines the spatial coordinates where fields are evaluated. The choice of grid depends on the geometry of the optical problem. Cartesian grids are appropriate for rectangular systems, while polar grids are natural for circularly symmetric optics such as telescopes and lenses.

### Cartesian Grids

Cartesian grids use uniform spacing in x and y coordinates. The `make_uniform_grid` function creates such a grid by specifying the number of points in each dimension and the total extent. This is the most common grid type for general-purpose optical simulations.

In [ ]:
# Create a uniform Cartesian grid
# 32x32 points spanning a 2x1 unit rectangle
grid = make_uniform_grid([32, 32], [2, 1])

print(f"Grid dimensions: {grid.dims}")
print(f"Grid spacing: {grid.delta}")
print(f"Total extent: {grid.delta[0] * grid.dims[0]:.4f} x {grid.delta[1] * grid.dims[1]:.4f}")

plt.figure(figsize=(10, 6))
plt.plot(grid.x, grid.y, '+', markersize=8, alpha=0.6)
plt.xlabel('x')
plt.ylabel('y')
plt.title('Cartesian Grid Sampling Pattern')
plt.grid(True, alpha=0.3)
plt.gca().set_aspect('equal')
plt.show()

The grid object stores coordinates efficiently. For regularly spaced grids, HCIPy does not store every coordinate explicitly but rather computes them on demand from the spacing, dimensions, and origin. This reduces memory usage while maintaining fast access.

### Polar Grids

For optical systems with circular symmetry, polar coordinates (radius r and angle theta) are often more appropriate. Polar grids can use different sampling strategies in the radial direction, such as logarithmic spacing to provide higher resolution near the center where optical features may be most significant.

HCIPy enforces type safety between coordinate systems. A polar grid only provides r and theta coordinates; attempting to access x or y directly will raise an error. This prevents coordinate system confusion in complex simulations.

In [ ]:
# Create a polar grid with logarithmic radial spacing
# This provides finer sampling near the origin
r = np.logspace(-1, 1, 11)
theta = np.linspace(0, 2 * np.pi, 10, endpoint=False)

coords = SeparatedCoords((r, theta))
polar_grid = PolarGrid(coords)

# Convert to Cartesian for visualization
cart_grid = polar_grid.as_('cartesian')

plt.figure(figsize=(8, 8))
plt.plot(cart_grid.x, cart_grid.y, '+', markersize=10)
plt.axis('equal')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Polar Grid (Logarithmic Radial Spacing)')
plt.grid(True, alpha=0.3)
plt.show()

print(f"Polar grid size: {polar_grid.size}")
print(f"Radial range: {polar_grid.r.min():.2f} to {polar_grid.r.max():.2f}")

Notice the denser sampling near the center in the polar grid. This logarithmic spacing is particularly useful for problems where the origin requires higher resolution, such as certain coronagraph designs or fiber coupling simulations.

## Fields

A `Field` associates values with each point on a grid. Fields are the primary data structure in HCIPy for representing physical quantities. The field stores both the numerical values and a reference to the grid, ensuring that the data always knows its spatial context.

### Real Fields

Real-valued fields typically represent intensity or amplitude distributions. They are straightforward NumPy arrays wrapped with grid information.

In [ ]:
# Create a grid for field examples
pupil_grid = make_pupil_grid(128, diameter=1.5)

# Real field: Gaussian intensity pattern
# This could represent a laser beam profile or point spread function
x = pupil_grid.x
y = pupil_grid.y
gaussian_values = np.exp(-(x**2 + y**2) / 0.3)
field = Field(gaussian_values, pupil_grid)

plt.figure(figsize=(10, 4))

plt.subplot(121)
imshow_field(field, cmap='inferno')
plt.colorbar(label='Intensity')
plt.title('Gaussian Field (2D)')

plt.subplot(122)
center_slice = field.shaped[field.shaped.shape[0]//2, :]
plt.plot(pupil_grid.x.reshape(pupil_grid.dims)[0, :], center_slice)
plt.xlabel('x')
plt.ylabel('Intensity')
plt.title('Cross-Section')
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Field size: {field.size} points")

The `shaped` property provides access to the field as a 2D array, which is useful for slicing and other array operations. The field maintains its connection to the grid regardless of how the data is accessed.

### Complex Fields

Complex fields are essential in wave optics. They represent the electric field with both amplitude and phase information. In HCIPy, the electric field is represented as E = A * exp(i*phi), where A is the amplitude and phi is the phase.

HCIPy provides automatic visualization of complex fields using a two-dimensional colorscale that simultaneously encodes amplitude (as brightness) and phase (as hue). This allows immediate visual assessment of wavefront properties.

In [ ]:
# Create a larger grid for the complex field example
grid = make_uniform_grid([128, 128], [5, 5])

x, y = grid.coords
r = grid.as_('polar').r

# Complex field: Gaussian amplitude with linear phase tilt
# The tilt represents a plane wave at an angle
z = np.exp(-r**2) * np.exp(-1j*(5 * x + 2 * y))
z = Field(z, grid)

plt.figure(figsize=(12, 5))

plt.subplot(131)
imshow_field(z)
plt.title('Complex Field (Amplitude + Phase)')

plt.subplot(132)
imshow_field(np.abs(z), cmap='inferno')
plt.title('Amplitude Only')

plt.subplot(133)
imshow_field(np.angle(z), vmin=-np.pi, vmax=np.pi, cmap='hsv')
plt.title('Phase Only')

plt.tight_layout()
plt.show()

print(f"Field is complex: {np.iscomplexobj(z)}")
print(f"Phase range: {np.min(np.angle(z)):.2f} to {np.max(np.angle(z)):.2f} radians")

The phase pattern in the right panel shows the linear tilt, which would cause the beam to propagate at an angle. The phase wraps around at ±π, creating the color discontinuities visible in the plot.

## Field Generators

Field generators are functions that create fields on any given grid. They are particularly useful for defining reusable optical elements such as apertures, where the same functional form needs to be evaluated on different grids.

A field generator takes a grid as input and returns a field sampled on that grid. This abstraction allows optical elements to be defined once and used throughout a simulation, regardless of the specific grid being used.

In [ ]:
# Define aperture generators
# These are functions that can be called with any grid
circular_aperture = make_circular_aperture(diameter=1.0)
rectangular_aperture = make_rectangular_aperture([1.0, 0.5])
hexagonal_aperture = make_hexagonal_aperture(1.0)

# Evaluate on the grid
circ_pupil = circular_aperture(pupil_grid)
rect_pupil = rectangular_aperture(pupil_grid)
hex_pupil = hexagonal_aperture(pupil_grid)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

imshow_field(circ_pupil, ax=axes[0], cmap='gray')
axes[0].set_title('Circular Aperture')
axes[0].axis('off')

imshow_field(rect_pupil, ax=axes[1], cmap='gray')
axes[1].set_title('Rectangular Aperture')
axes[1].axis('off')

imshow_field(hex_pupil, ax=axes[2], cmap='gray')
axes[2].set_title('Hexagonal Aperture')
axes[2].axis('off')

plt.tight_layout()
plt.show()

The field generator abstraction is powerful because it separates the definition of an optical element from its sampling. The same circular aperture function can be used on a coarse grid for quick calculations or a fine grid for high-accuracy simulations.

## Supersampling

When evaluating apertures with sharp edges, standard sampling can introduce aliasing artifacts. This occurs because a sharp edge contains high spatial frequencies that may not be adequately captured by the grid sampling.

Supersampling addresses this by evaluating the aperture function at a higher resolution and then downsampling. This is equivalent to computing the average value over each pixel, producing more accurate results at edges.

The following example demonstrates this with the Magellan telescope aperture, which has thin spider arms that are particularly susceptible to aliasing.

In [ ]:
# Magellan telescope aperture with thin spider arms
magellan = make_magellan_aperture(normalized=True)

# Direct sampling
pupil_direct = magellan(pupil_grid)

# 8x supersampling
pupil_supersampled = evaluate_supersampled(magellan, pupil_grid, 8)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

imshow_field(pupil_direct, ax=axes[0], cmap='gray')
axes[0].set_title('Direct Sampling (Aliasing on spider arms)')

imshow_field(pupil_supersampled, ax=axes[1], cmap='gray')
axes[1].set_title('8x Supersampling (Smooth edges)')

plt.tight_layout()
plt.show()

# Zoomed comparison to highlight the difference
print("Zoomed view of spider arm region:")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

imshow_field(pupil_direct, ax=axes[0], cmap='gray')
axes[0].set_title('Direct Sampling (zoomed)')
axes[0].set_xlim(-0.05, 0.25)
axes[0].set_ylim(-0.05, 0.25)

imshow_field(pupil_supersampled, ax=axes[1], cmap='gray')
axes[1].set_title('8x Supersampling (zoomed)')
axes[1].set_xlim(-0.05, 0.25)
axes[1].set_ylim(-0.05, 0.25)

plt.tight_layout()
plt.show()

The zoomed view clearly shows the difference: direct sampling produces jagged, pixelated edges on the spider arms, while supersampling yields smooth, accurate representations. For high-contrast imaging applications where precise aperture definition is critical, supersampling is essential.

## Grid Weights

Each grid point in HCIPy has an associated weight representing the area (in 2D) or volume (in 3D) that it represents. These weights enable numerical integration over the grid.

For uniformly spaced Cartesian grids, all weights are equal. For non-uniform or unstructured grids, weights vary to properly account for the differing areas represented by each point.

The following example demonstrates computing the area of a circular aperture by integration, which should equal πr².

In [ ]:
# Examine grid weights
print(f"Individual point weight: {pupil_grid.weights:.6f} square units")
print(f"Total grid area: {pupil_grid.weights.sum():.4f} square units")
print(f"Grid dimensions: {pupil_grid.dims}")

# Compute aperture area via weighted integration
# This is equivalent to the integral: integral of A(x,y) dx dy
aperture = make_circular_aperture(diameter=1.0)(pupil_grid)
computed_area = (aperture * pupil_grid.weights).sum()
expected_area = np.pi * (0.5)**2

print(f"\nCircular Aperture Area Calculation:")
print(f"  Computed area: {computed_area:.6f}")
print(f"  Expected area (πr²): {expected_area:.6f}")
print(f"  Relative error: {abs(computed_area - expected_area)/expected_area * 100:.3f}%")

The small discrepancy between computed and expected area arises from the finite grid resolution. As the grid spacing decreases, the computed area converges to the exact value. This demonstrates how grid weights transform discrete sums into continuous integrals.

## Summary

This tutorial has introduced the fundamental components of HCIPy:

**Grids**
* Cartesian grids provide uniform sampling for general-purpose simulations
* Polar grids offer natural coordinates for circularly symmetric systems
* Type-safe coordinate access prevents coordinate system confusion
* Grid transformations allow conversion between coordinate systems

**Fields**
* Real fields represent intensity or amplitude distributions
* Complex fields encode both amplitude and phase for wave optics
* Automatic visualization handles appropriate colorscaling
* Fields maintain connection to their defining grid

**Field Generators**
* Create reusable optical element definitions
* Evaluate consistently on any grid
* Standard library includes common apertures and optics

**Numerical Techniques**
* Supersampling eliminates aliasing at sharp edges
* Grid weights enable accurate numerical integration

These concepts form the foundation for more advanced topics including optical propagation, field interpolation, and atmospheric turbulence simulation.